# 5G Network Intelligence — Pretrained Zero-Shot Forecasting

## Hypothesis and provenance limits

Can a fixed pretrained forecaster use the 168-hour target history better than the current development references? Test one pinned Chronos-Bolt Base checkpoint and its native median; do not fine-tune or select quantiles. Model-card license: Apache-2.0. The pinned weights appear in public history before the target dataset's first visible commit, but the full pretraining corpus has not been audited. This is not a no-leakage certification or an unseen-test claim. [Author model card](https://huggingface.co/amazon/chronos-bolt-base).

## Temporal protocol

Reuse development scoring windows after 383 through 443 and after 443 through 503. Forecast all original beams from exactly 168 observed target values. No validation 504–671, original final holdout or production promotion.


In [ ]:
RUN_INFERENCE = False
SOURCE_COMMIT = 'f429182ca94361ed8beb35cbb2ac3a0fb0244bf4'


## Fixed configuration

One checkpoint, FP32 GPU inference, batch 256 and quantile 0.5. All weights are frozen. Negative point forecasts are clipped to zero as declared before scoring. No data-dependent calibration, model search or local weight download.


In [ ]:
PROTOCOL = {'protocol': 'chronos-bolt-development-zero-shot-v1',
 'model_id': 'amazon/chronos-bolt-base',
 'revision': '5d9f166d69f47aef3401367a7b842e78fe97b121',
 'weights_sha256': '31f875483a3215bc6880a0837ea608a13ce55f88ad90538c3cdd0b29aeb60b36',
 'chronos_version': '2.3.2',
 'transformers_version': '4.57.6',
 'context_length': 168,
 'prediction_length': 24,
 'horizons': [1, 6, 24],
 'quantile': 0.5,
 'batch_size': 256,
 'dtype': 'float32',
 'seed': 17,
 'folds': [{'name': 'inner1', 'history_boundary': 383, 'score_end': 443},
           {'name': 'inner2', 'history_boundary': 443, 'score_end': 503}],
 'missing_inputs': 'preserve_NaN_with_native_model_mask',
 'zeros': 'observed_values_not_missing',
 'postprocessing': 'clip_negative_median_forecasts_to_zero',
 'fine_tuning': 'forbidden',
 'coefficient_selection': 'forbidden',
 'validation_504_671': 'forbidden',
 'original_final_holdout': 'forbidden',
 'pretraining_overlap': 'not_certified_absent',
 'production_promotion': 'forbidden'}


## Input window and missingness

The next cell defines target-only history slicing. Observed zeros remain zeros; missing values remain NaN for the model's native mask. No future row or non-target KPI enters the context.


In [ ]:
"""Causal target-only windows, with zeros and missingness preserved."""

import numpy as np


def target_context(values, origin, beams):
    values=np.asarray(values)
    beams=np.asarray(beams)
    if (values.ndim!=3 or values.shape[2]!=4 or not isinstance(origin,(int,np.integer))
            or origin<167 or origin>=len(values) or beams.ndim!=1 or not beams.size
            or not np.issubdtype(beams.dtype,np.integer) or beams.min()<0 or beams.max()>=values.shape[1]):
        raise ValueError("Valid absolute origin, beam indices and 168 historical rows required")
    result=values[origin-167:origin+1,beams,0].T.astype(np.float32,copy=True)
    if np.isinf(result).any():
        raise ValueError("Infinite input is invalid; missing observations must use NaN")
    return result


## Hosted setup and audited source

Attach the canonical completed hurdle output and the project zero-shot source dataset. The pretrained library wheel is downloaded from its recorded public PyPI URL and verified against its SHA256 before installation. Model weights download only inside this explicitly enabled hosted worker.


In [ ]:
from pathlib import Path
import os,sys,json,shutil,subprocess,tarfile,hashlib
from urllib.request import urlopen
if RUN_INFERENCE:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Hosted Kaggle required')
    roots=[p.parent.parent for p in Path('/kaggle/input').rglob('train_hurdle_research.py') if (p.parent.parent/'data/processed/panel.npz').exists()]
    if len(roots)!=1:
        raise RuntimeError('Attach exactly one canonical source output')
    PROJECT=Path('/kaggle/working/chronos_zero_shot')
    PROJECT.mkdir(exist_ok=True)
    for folder in ('src','configs','data/processed'):
        shutil.copytree(roots[0]/folder,PROJECT/folder,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
    os.chdir(PROJECT)
    sys.path.insert(0,str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT']=SOURCE_COMMIT
    for folder in ('scripts','reports'):
        Path(folder).mkdir(exist_ok=True)
    Path('scripts/__init__.py').touch()
    patches=list(Path('/kaggle/input').rglob('evaluate_chronos_zero_shot.py'))
    if len(patches)!=1:
        raise RuntimeError('Attach exactly one zero-shot source')
    patch=patches[0].parent
    for name,folder in (('evaluate_chronos_zero_shot.py','scripts'),('zero_shot_context.py','src/features'),('chronos_zero_shot.yaml','configs'),('pretrained_forecaster_source_audit.json','reports')):
        shutil.copyfile(patch/name,Path(folder)/name)
    audit=json.loads((patch/'pretrained_forecaster_source_audit.json').read_text())
    wheel=audit['library_distribution']
    wheel_path=PROJECT/wheel['filename']
    with urlopen(wheel['url'],timeout=60) as response:
        wheel_path.write_bytes(response.read())
    if hashlib.sha256(wheel_path.read_bytes()).hexdigest()!=wheel['sha256']:
        raise RuntimeError('Audited library wheel checksum mismatch')
    os.environ['USE_TF']='0'
    os.environ['USE_FLAX']='0'
    subprocess.run([sys.executable,'-m','pip','install',str(wheel_path),'transformers=='+audit['transformers_version_pin']],check=True)


## Frozen GPU inference and debugging

The evaluator verifies the safetensors hash, disables gradients, keeps FP32 and saves a forecast cache after each origin. No optimizer or fitting operation occurs. The source checkpoint remains separately in Kaggle Output; the result archive contains project reports and forecasts, not copied model weights.


In [ ]:
if RUN_INFERENCE:
    with Path('reports/chronos_zero_shot.log').open('w') as log:
        process=subprocess.Popen([sys.executable,'-u','-m','scripts.evaluate_chronos_zero_shot'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line)
            log.flush()
        status=process.wait()
    with tarfile.open('/kaggle/working/chronos_zero_shot_results.tar.gz','w:gz') as archive:
        for folder in ('reports','artifacts/chronos_zero_shot'):
            if Path(folder).exists():
                archive.add(folder,arcname=folder)
    if status:
        raise RuntimeError('Zero-shot job failed; inspect saved log and partial cache')
else:
    print('Review mode: no model download, inference, fitting or calibration.')


## Saved metrics and limitations

MAE is in original usage units, WAPE is a ratio and every declared horizon retains its full original population. Scores remain exploratory because development has been reused and pretraining overlap is not certified absent. No improvement or calibrated uncertainty is assumed from the publisher's benchmark claims.


In [ ]:
paths=[Path('reports/chronos_zero_shot_development.csv'),Path('../reports/chronos_zero_shot_development.csv')]
if Path('/kaggle/input').is_dir():
    paths+=list(Path('/kaggle/input').rglob('chronos_zero_shot_development.csv'))
path=next((p for p in paths if p.exists()),None)
if path:
    import pandas as pd
    from IPython.display import display
    display(pd.read_csv(path))
else:
    print('No completed zero-shot scores attached.')


## Completed v1 finding and decision

The checkpoint improved 6h development MAE by 0.458% / 0.283% across the two folds, but 1h and 24h regressed in both. WAPE remained 40.51–47.08%, and zero false-positive rates were 99.81–99.93%. All 127 artifact files and mapping of 875,520 scored forecasts to their origin caches were checked; this did not independently execute the external model or certify its corpus. Do not promote this full-range standalone configuration or tune it on reused validation.
